# Dependency Management — Bob Discovers the Force of uv

*One reformed admin, one project, three files: `pyproject.toml`, `uv.lock`, `.venv`.*

## Introduction — Bob Checks the Dependencies

> **Bob:** My first uv project has its own Python, a `pyproject.toml`, and a build. I put all three in the runbook.
>
> **Bob:** Now Alice says the same tool can keep my dependencies predictable. The Force of uv? Let's see what happens when I change a version.

This notebook follows [Bob's first uv project](../uv_proj_scaffolding/uv_proj_scaffolding_lab.ipynb). Run it in its own Dev Container; it starts a separate dependency-focused example in `/workspace`. Three files carry the story:

| File | Role |
| --- | --- |
| `pyproject.toml` | **Intent** — which packages, which version ranges |
| `uv.lock` | **Truth** — the fully resolved graph with exact versions |
| `.venv` | **Reality** — what is actually installed |

## Act I — Start with a Project

### What Is on the Machine?

> **Bob:** First rule of any runbook: know your starting point.

The dev container ships `uv` preinstalled. Verify the version:

In [ ]:
%%bash
set -euxo pipefail
uv --version

### Remove Old Experiments

Remove files generated by a previous run of this dedicated lab (including the example Git helper), so the commands start from a known state:

In [ ]:
%%bash
set -euxo pipefail
rm -rf pyproject.toml uv.lock .venv local-helper requirements.txt

## Act II — Declare Dependencies

### Bob Writes a `pyproject.toml`

> **Bob:** Alright. No `pip install` into anything. First, I write down what the project *needs*.

The `pyproject.toml` declares intent, not exact installs. `httpx` is exact-pinned on purpose — the upgrade story later needs it. Create the file:

In [ ]:
%%bash
set -euxo pipefail
cat > pyproject.toml <<'EOF'
[project]
name = "bobs-license-service"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.115,<0.116",
    "httpx==0.24.1",
    "rich>=13.7,<14",
]
EOF

Read the file back before doing anything with it:

In [ ]:
%%bash
set -euxo pipefail
cat pyproject.toml

### "Or I Could Just Ask uv"

> **Bob:** Wait — I hand-edited a config file like it's 2010. Alice's note says `uv add` does this for me.

`uv add` writes the constraint into `pyproject.toml`, resolves the graph, and installs — all in one step. Add `requests`:

In [ ]:
%%bash
set -euxo pipefail
uv add requests

> **Bob:** It edited my file, locked, *and* installed. Suspiciously convenient.

Removing works the same way. `uv remove` deletes the entry and cleans the package out of the environment:

In [ ]:
%%bash
set -euxo pipefail
uv remove requests

### Dependency Groups

> **Bob:** My test tools don't belong on the production server. There has to be a drawer for those.

There is: `[dependency-groups]`. Add `pytest` to a `dev` group — it lands in the group, not in `dependencies`:

In [ ]:
%%bash
set -euxo pipefail
uv add --group dev pytest

Inspect what `uv` wrote — a new `[dependency-groups]` table, separate from the runtime dependencies:

In [ ]:
%%bash
set -euxo pipefail
cat pyproject.toml

### Dependency Markers

> **Bob:** One of my scripts needs `pywin32` — but only on the Windows box. On this Linux machine that package doesn't even exist.

A PEP 508 marker makes the dependency conditional. Rewrite the file with a platform-guarded `pywin32`:

In [ ]:
%%bash
set -euxo pipefail
cat > pyproject.toml <<'EOF'
[project]
name = "bobs-license-service"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.115,<0.116",
    "httpx==0.24.1",
    "rich>=13.7,<14",
    "pywin32>=310; sys_platform == 'win32'",
]

[dependency-groups]
dev = [
    "pytest>=8.0",
]
EOF

Resolution still succeeds on Linux: the resolver records `pywin32` for Windows environments but skips it here. Without the marker, this command would fail:

In [ ]:
%%bash
set -euxo pipefail
uv lock

## Act III — Lock and Sync

### `uv lock` — Resolving the Graph

> **Bob:** So `uv.lock` is the exact bill of materials. My file says "any fastapi 0.115.x" — the lockfile says which one, and every transitive package too.

The lock already happened above; confirm the file exists and see its size:

In [ ]:
%%bash
set -euxo pipefail
ls -la uv.lock

### `uv tree` — What Did I Just Get?

Three declared dependencies pull in a whole graph of transitive ones. Display it:

In [ ]:
%%bash
set -euxo pipefail
uv tree

### `uv sync` — Making the Environment Match

> **Bob:** Lock is truth, `.venv` is reality. Time to make reality match.

`uv sync` installs, upgrades, or removes packages until `.venv` matches `uv.lock` exactly:

In [ ]:
%%bash
set -euxo pipefail
uv sync

## Act IV — Change a Requirement

### Bumping a Pin

> **Bob:** Security bulletin says update `httpx`. Old me would have run `pip install --upgrade` on the server and hoped. New me edits the intent first.

Change the exact pin in `pyproject.toml`:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.24.1/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

The file changed, but `uv.lock` still records the old version. Re-resolve so the lockfile reflects the new pin:

In [ ]:
%%bash
set -euxo pipefail
uv lock

Then make the environment follow. These commands are separate so Bob can inspect each effect; `uv sync` can also re-lock automatically:

In [ ]:
%%bash
set -euxo pipefail
uv sync

### A Broken Pin Fails Early

> **Bob:** I have a bad feeling about this version pin. What happens if I ask for one that doesn't exist?

Write an impossible pin:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.27.2/httpx==999.0.0/' pyproject.toml
grep -n 'httpx' pyproject.toml

Resolution fails immediately — at the desk, not in production. The `|| true` keeps the notebook running past the expected error:

In [ ]:
%%bash
set -euxo pipefail
uv lock || true

### Repairing the Pin

Restore the valid version:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==999.0.0/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

Re-resolve to confirm the repair:

In [ ]:
%%bash
set -euxo pipefail
uv lock

## Act V — Check for Drift

### Drift Detection

> **Bob:** Suppose a colleague edits `pyproject.toml` and forgets to re-lock. Intent and truth drift apart — silently. I want an alarm for that.

Simulate the forgetful colleague — edit the `rich` constraint without re-locking:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.8,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

`uv lock --check` compares intent against truth and fails on drift. This is the command for CI and pre-commit:

In [ ]:
%%bash
set -euxo pipefail
uv lock --check || true

Revert the edit:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.8,<14/rich>=13.7,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

Now the check passes again:

In [ ]:
%%bash
set -euxo pipefail
uv lock --check

### Locked Installs

> **Bob:** In CI I want zero cleverness. Install exactly what was reviewed, or fail loudly.

Create drift again — change `rich` without re-locking:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.9,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

A plain `uv sync` can re-lock. `uv sync --locked` checks whether `uv.lock` is current and fails on drift (unlike `--frozen`, which skips that check):

In [ ]:
%%bash
set -euxo pipefail
uv sync --locked || true

Refresh the lockfile the explicit way:

In [ ]:
%%bash
set -euxo pipefail
uv lock

With intent and lockfile aligned, the locked sync succeeds:

In [ ]:
%%bash
set -euxo pipefail
uv sync --locked

## Act VI — Upgrade on Bob's Terms

### Exact Pins Don't Move

> **Bob:** Maintenance day. Let's see who decides what gets upgraded — me, or the tool.

Pin `fastapi` to one exact version:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.115,<0.116/fastapi==0.115.0/' pyproject.toml
grep -n 'fastapi' pyproject.toml

Ask for an upgrade of that one package. Nothing moves — the constraint allows exactly one version, and the resolver respects it:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade-package fastapi

### Ranges Move Within Bounds

Loosen the constraint back to a range:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi==0.115.0/fastapi>=0.115,<0.116/' pyproject.toml
grep -n 'fastapi' pyproject.toml

The same command now refreshes `fastapi` to the newest version inside the range — and touches nothing else:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade-package fastapi

### The Maintenance Pass

> **Bob:** And once a month: everything to the newest version my ranges allow, in one reviewable lockfile change.

Upgrade all dependencies within their declared bounds:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade

## Act VII — Export and Explore Sources

> **Bob:** Some colleagues still need a `requirements.txt`. I can give them one without abandoning my lockfile.

Export the project's resolved dependencies for a workflow that expects `requirements.txt`:

In [ ]:
%%bash
set -euxo pipefail
uv export --format requirements.txt -o requirements.txt --quiet

Inspect the start of the exported file. Regenerate it whenever project dependencies change:

In [ ]:
%%bash
set -euxo pipefail
sed -n '1,18p' requirements.txt

The export is for compatibility; `pyproject.toml` and `uv.lock` remain the source of truth. For legacy environments, `uv pip`, and standalone tools, see [Chapter 03, Section 04](../../docs/chapter-03/section-04.md).

For an alternative package source, uv can add a Git repository. The next example uses a local Git checkout so it works independently of GitHub. First, create a small library outside the project (to avoid automatically joining a workspace):

In [ ]:
%%bash
set -euxo pipefail
uv init --lib --no-workspace --vcs none --author-from none --python 3.11 local-helper

Commit the helper so uv has a Git revision to resolve:

In [ ]:
%%bash
set -euxo pipefail
git -C local-helper -c user.name=Bob -c user.email=bob@example.invalid init -q && git -C local-helper add . && git -C local-helper -c user.name=Bob -c user.email=bob@example.invalid commit -qm 'Add local helper'

Add the helper from its Git checkout. `uv add` records a dependency in `[project]` and its Git source in `[tool.uv.sources]`:

In [ ]:
%%bash
set -euxo pipefail
uv add "local-helper @ git+file://$(pwd)/local-helper"

Check the entries uv wrote. The repository location is separate from the package name:

In [ ]:
%%bash
set -euxo pipefail
sed -n '1,55p' pyproject.toml

Remove the demonstration dependency so the main example stays focused:

In [ ]:
%%bash
set -euxo pipefail
uv remove local-helper

Confirm the dependency graph is up to date after removal:

In [ ]:
%%bash
set -euxo pipefail
uv lock --check

## Epilogue — Bob's Runbook Notes

> **Bob:** New page in the runbook. Alice gets a coffee for the sticky note.

### File, Command, Effect

| File | Changed by | Read by |
| --- | --- | --- |
| `pyproject.toml` | you, `uv add`, `uv remove` | `uv lock` |
| `uv.lock` | `uv lock`, `uv add`, `uv remove` | `uv sync` |
| `.venv` | `uv sync` | your application |

### Bob's Three Rules

1. **Declare intent, don't install by hand.** Dependencies go through `uv add` into `pyproject.toml` — never `pip install` straight into an environment.
2. **Check before commit.** `uv lock` records resolved versions, `uv sync` applies them, and `uv lock --check` catches drift.
3. **Locked in CI.** `uv sync --locked` checks the lockfile before installing. Use `uv export` if another workflow needs a requirements file.